# BurstGPT 3: training and scenario comparisons
Run the selected traffic ranges through the shared simulator and DQN pipeline, then share charts and an HTML report with the team.

Use a **CPU runtime** (Runtime → Change runtime type → CPU). A GPU is not required: GPU service is simulated. The full test set has over 1.8 million valid requests and will take substantially longer than the earlier small smoke test.

Workflow: setup → load your CSV → inspect the dataset → train/select on validation → optionally run final frozen tests → charts → export.

The second annotated range is called **mixed traffic**, because it contains substantial peaks. The third is **peak heavy**. The first range is split into training and validation with a five-minute gap. No traffic-regime filter is applied.

The initial default is one seed and 24 **sampled episodes**, not a pass through all 1,297 available training episodes. Increase the budget and use multiple seeds for final experiments. This notebook evaluates a **frozen** policy; it does not implement continuous adaptation.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "burstgpt_3"  # Change to main after this branch is merged.
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    REPO = Path("/content/llm-request-triage")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "status", "--short"], check=True)
        print("Using the existing checkout. Fetch/switch explicitly if you need another revision.")
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent

required = ["experiments/evaluate_scenarios.py", "configs/burstgpt3_ranges.json",
            "scripts/prepare_burstgpt3.py", "src/llm_triage/workload/time_ranges.py"]
assert all((REPO / name).is_file() for name in required), "Checkout lacks BurstGPT 3 changes; use the pushed burstgpt_3 branch."
os.chdir(REPO)
assert sys.version_info >= (3, 11), "Use Python 3.11 or newer."
print("Repository:", REPO)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,rl,analysis]", "pandas>=2.0"], check=True)
sys.path.insert(0, str(REPO / "src"))

import base64
import csv
from datetime import datetime, timezone
import hashlib
import html
import importlib.metadata
import json
import shutil
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

from llm_triage.workload.burstgpt import sha256


## Run settings
Change these settings **before training**. A blank run name creates a unique output folder. Rerun with the same name only to resume the same experiment; changing data or model settings requires a fresh run.

`CSV_MODE="drive"` mounts Google Drive and copies your CSV to Colab's local disk for faster repeated reads. Alternatively choose `upload`. The CSV is approximately 221 MB and is not stored in GitHub.

Changing the episode duration changes where simulator queues reset. It does not change the selected timestamp intervals.


In [ ]:
CSV_MODE = "drive"  # @param ["drive", "upload"]
DRIVE_CSV_PATH = "/content/drive/MyDrive/BurstGPT_3.csv"  # @param {type:"string"}
LOCAL_CSV_PATH = str(REPO / "BurstGPT_3.csv")  # Used when running locally.
RUN_NAME = ""  # @param {type:"string"}
SEEDS = [7]  # For final experiments consider [7, 17, 27].
EPISODES = 24  # @param {type:"integer"}
VALIDATE_EVERY = 12  # @param {type:"integer"}
EPISODE_SECONDS = 300  # @param {type:"integer"}
TRAIN_TARGET_RPS = [None]  # None preserves the original traffic timing.
RUN_REPOSITORY_TESTS = True  # @param {type:"boolean"}

run_name = RUN_NAME or datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
assert Path(run_name).name == run_name and run_name not in {".", ".."}, "Use a simple folder name."
RUN_ROOT = REPO / "results" / "runs" / "colab_burstgpt3" / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES = RUN_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
DATA_DIR = RUN_ROOT / "data"
ANALYSIS_DIR = RUN_ROOT / "traffic_analysis"
TRAIN_OUTPUT = RUN_ROOT / "training_run"
TEST_ROOT = RUN_ROOT / "test_scenarios"
COMMAND_LOG = RUN_ROOT / "notebook_commands.log"
RANGE_CONFIG_PATH = RUN_ROOT / "ranges.json"
TRAIN_CONFIG_PATH = RUN_ROOT / "training_config.json"
test_results = {}
generated_figures = set()
print("Run folder:", RUN_ROOT)

def run_command(args):
    command = [str(value) for value in args]
    print("Running:", " ".join(command), flush=True)
    with COMMAND_LOG.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def save_figure(fig, name):
    path = FIGURES / (name + ".png")
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    generated_figures.add(path.name)
    display(Image(filename=str(path)))
    return path

def save_config(path, value):
    if path.exists():
        assert json.loads(path.read_text()) == value, "Existing run config differs. Use a new RUN_NAME."
    else:
        path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


In [ ]:
if IS_COLAB:
    SOURCE_CSV = REPO / "data" / "raw" / "BurstGPT_3.csv"
    SOURCE_CSV.parent.mkdir(parents=True, exist_ok=True)
    if not SOURCE_CSV.exists():
        if CSV_MODE == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            drive_source = Path(DRIVE_CSV_PATH)
            assert drive_source.is_file(), "Set DRIVE_CSV_PATH to your BurstGPT_3.csv."
            shutil.copyfile(drive_source, SOURCE_CSV)
        elif CSV_MODE == "upload":
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if Path(name).suffix.lower() == ".csv"]
            assert len(csv_files) == 1, "Upload exactly one BurstGPT CSV."
            SOURCE_CSV.write_bytes(uploaded[csv_files[0]])
            del uploaded
        else:
            raise ValueError("CSV_MODE must be drive or upload")
else:
    SOURCE_CSV = Path(LOCAL_CSV_PATH).expanduser().resolve()

assert SOURCE_CSV.is_file(), "BurstGPT_3.csv was not found."
print("CSV:", SOURCE_CSV, "Bytes:", SOURCE_CSV.stat().st_size)
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q"])


## Inspect and record the dataset
The ranges below use **original CSV timestamps in seconds**, with inclusive starts and exclusive ends. The red-box boundaries were estimated from the annotated plot and aligned to five-minute bins.

To change the research protocol, edit `range_config["ranges"]` here before running preparation. The prepared manifest records empty episodes and source/config checksums. The analysis's automatic 60/20/20 regime labels do not control this explicit-range dataset.

Never adjust the test ranges or checkpoint based on test scores. No-record intervals can mean inactivity or collection gaps; they are not proof of actual idle traffic.


In [ ]:
range_config = json.loads((REPO / "configs/burstgpt3_ranges.json").read_text())
range_config["episode_seconds"] = EPISODE_SECONDS
# Optional: edit range_config["ranges"] here before preparation.
assert sha256(SOURCE_CSV) == range_config["source_sha256"], "CSV does not match the selected BurstGPT 3 source."
save_config(RANGE_CONFIG_PATH, range_config)
display(pd.DataFrame(range_config["ranges"]))

if not (ANALYSIS_DIR / "summary.json").exists():
    run_command([sys.executable, "scripts/analyze_burstgpt_traffic.py", "--source", SOURCE_CSV,
                 "--output-dir", ANALYSIS_DIR, "--bin-seconds", 300])
analysis = json.loads((ANALYSIS_DIR / "summary.json").read_text())
assert analysis["source"]["sha256"] == range_config["source_sha256"]

if not (DATA_DIR / "processed" / "manifest.json").exists():
    run_command([sys.executable, "scripts/prepare_burstgpt3.py", "--source", SOURCE_CSV,
                 "--analysis-dir", ANALYSIS_DIR, "--ranges-config", RANGE_CONFIG_PATH, "--data-dir", DATA_DIR])
manifest = json.loads((DATA_DIR / "processed" / "manifest.json").read_text())
assert manifest["range_config"] == range_config, "Prepared ranges differ. Use a fresh run."
assert manifest["source_sha256"] == sha256(SOURCE_CSV)
assert manifest["selected_sha256"] == sha256(DATA_DIR / "processed" / "burstgpt_windows.csv")

episode_table = pd.DataFrame(manifest["windows"] + manifest.get("empty_windows", []))
episode_table["duration_s"] = episode_table["end_timestamp_s"] - episode_table["start_timestamp_s"]
episode_table["nonempty"] = episode_table["requests"] > 0
dataset_summary = episode_table.groupby(["split", "group"], sort=False).agg(
    requests=("requests", "sum"), episodes=("window", "size"),
    nonempty_episodes=("nonempty", "sum"), duration_s=("duration_s", "sum")).reset_index()
dataset_summary["no_record_episode_fraction"] = 1 - dataset_summary["nonempty_episodes"] / dataset_summary["episodes"]
dataset_summary["mean_rps_including_gaps"] = dataset_summary["requests"] / dataset_summary["duration_s"]
display(dataset_summary)
dataset_summary.to_csv(RUN_ROOT / "dataset_summary.csv", index=False)

traffic = pd.read_csv(ANALYSIS_DIR / "time_bins.csv")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(traffic["start_timestamp_s"], traffic["mean_rps"], linewidth=0.6, color="0.5")
colors = {"train": "#2166ac", "validation": "#4daf4a", "test": "#d95f02"}
for item in range_config["ranges"]:
    ax.axvspan(item["start_timestamp_s"], item["end_timestamp_s"], alpha=0.18,
               color=colors[item["split"]], label=item["group"])
ax.ticklabel_format(axis="x", style="plain", useOffset=False)
ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Five-minute requests/s",
       title="Selected traffic ranges — natural traffic mix retained")
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, "selected_traffic_ranges")


## Train and select using validation
The pipeline trains DQN, chooses checkpoints by **mean validation episode SLO attainment**, reloads them, and compares them with FCFS, length-aware, and static SLO-aware policies. Test periods are reserved.

A training episode is sampled with replacement; the episode budget is not an epoch count. Sparse episodes and repeated deferrals change the number of learning steps. With larger datasets, inspect coverage and learning curves rather than assuming 24 episodes is sufficient.

The pipeline log prints its training-log path. Detailed checkpoint progress is written there while the subprocess runs. An existing run is resumed only with unchanged settings, code, and runtime.


In [ ]:
train_config = json.loads((REPO / "configs/burstgpt3_initial.json").read_text())
train_config.update(seeds=SEEDS, episodes=EPISODES, validate_every=VALIDATE_EVERY, target_rps=TRAIN_TARGET_RPS)
save_config(TRAIN_CONFIG_PATH, train_config)
display(train_config)
command = [sys.executable, "-m", "experiments.run_pipeline", "--config", TRAIN_CONFIG_PATH,
           "--data-dir", DATA_DIR, "--output", TRAIN_OUTPUT]
if (TRAIN_OUTPUT / "pipeline.json").exists():
    command.append("--resume")
run_command(command)
pipeline_state = json.loads((TRAIN_OUTPUT / "pipeline.json").read_text())
assert pipeline_state["status"] == "completed"
TRAINING_DIR = TRAIN_OUTPUT / "training"
training_record = json.loads((TRAINING_DIR / "run.json").read_text())
assert training_record["test_evaluated"] is False
checkpoint_hashes = {f"dqn_seed{item['seed']}.npz": item["sha256"] for item in training_record["checkpoints"]}
for filename, expected in checkpoint_hashes.items():
    assert sha256(TRAINING_DIR / filename) == expected
display(pd.DataFrame(training_record["checkpoints"]))


In [ ]:
history = pd.read_csv(TRAINING_DIR / "training.csv")
selection = pd.read_csv(TRAINING_DIR / "checkpoints.csv")
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for seed in history["seed"].unique():
    h = history[history["seed"] == seed]
    s = selection[selection["seed"] == seed]
    axes[0, 0].plot(s["episode"], s["validation_attainment"], marker="o", label=f"seed {seed}")
    axes[0, 1].plot(h["episode"], h["train_attainment"], label=f"seed {seed}")
    axes[1, 0].plot(h["episode"], h["reward"], label=f"seed {seed}")
    axes[1, 1].plot(h["episode"], h["loss"], label=f"seed {seed}")
for item in training_record["checkpoints"]:
    axes[0, 0].scatter(item["episode"], item["validation_attainment"], marker="*", s=150, color="black")
axes[0, 0].set_title("Validation attainment; stars mark selected checkpoints")
axes[0, 1].set_title("Training attainment (exploration enabled)")
axes[1, 0].set_title("Episode return (affected by request count)")
axes[1, 1].set_title("Mean training loss")
for ax in axes.flat:
    ax.set_xlabel("Training episode")
    ax.grid(alpha=0.2)
    ax.legend()
for ax in axes[0]:
    ax.yaxis.set_major_formatter(PercentFormatter(1))
save_figure(fig, "learning_curves")
coverage = history.groupby("seed").agg(sampled_episodes=("episode", "size"), unique_windows=("window", "nunique"))
display(coverage)


## Validation comparison
The pipeline's `summary.csv` averages episode scores. Since episodes have different request counts, the chart below also computes **request-weighted** attainment and drop rate. Keep these definitions separate when reporting results.


In [ ]:
validation_runs = pd.read_csv(TRAIN_OUTPUT / "validation.csv")
validation_runs["dropped_estimate"] = validation_runs["drop_rate"] * validation_runs["arrivals"]
validation_summary = validation_runs.groupby("policy", sort=False).agg(
    requests=("arrivals", "sum"), slo_met=("slo_met", "sum"),
    dropped=("dropped_estimate", "sum"), mean_episode_attainment=("slo_attainment", "mean")).reset_index()
validation_summary["overall_slo_attainment"] = validation_summary["slo_met"] / validation_summary["requests"]
validation_summary["overall_drop_rate"] = validation_summary["dropped"] / validation_summary["requests"]
display(validation_summary)
validation_summary.to_csv(RUN_ROOT / "validation_weighted_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, title in zip(axes, ["overall_slo_attainment", "overall_drop_rate"],
                             ["Validation: request-weighted SLO attainment", "Validation: request-weighted drop rate"]):
    ax.bar(validation_summary["policy"], validation_summary[metric])
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(axis="x", rotation=25)
    ax.set_title(title)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.2)
save_figure(fig, "validation_comparison")


## Final frozen test scenarios
Run this section after choosing the model settings on validation. Set `RUN_FINAL_TESTS=True` to evaluate both named test groups with the same frozen checkpoints. It is initially false so exploratory training does not automatically consume the final benchmark.

Native timing compares the naturally occurring mixed and peak-heavy traffic. Optional numeric rates provide controlled load tests while preserving request order and sizes; they rescale each episode independently.

The same requests are paired across policies. Weights are not updated, exploration is disabled, and test rewards are not fed into training. Online adaptation would require a separate experiment.

Full tests cover thousands of episodes and may take a while on Colab CPU. Do not interpret the current small training budget as a final optimized model.


In [ ]:
RUN_FINAL_TESTS = False  # @param {type:"boolean"}
EXTRA_TEST_RATES = []  # Optional example: [1.2, 2.4, 4.8]. Native timing always runs too.

if RUN_FINAL_TESTS:
    load_cases = [("native", None)] + [(f"rps_{rate:g}", [rate]) for rate in EXTRA_TEST_RATES]
    for label, rates in load_cases:
        output = TEST_ROOT / label
        if (output / "run.json").exists():
            saved = json.loads((output / "run.json").read_text())
            expected_rates = [None] if rates is None else rates
            assert saved["config"]["target_rps"] == expected_rates
            assert saved["checkpoints"] == checkpoint_hashes
            assert saved["manifest_sha256"] == sha256(DATA_DIR / "processed" / "manifest.json")
            assert saved["split"] == "test" and saved["groups"] is None
            print("Reusing completed evaluation:", output)
        else:
            command = [sys.executable, "-m", "experiments.evaluate_scenarios",
                       "--training-dir", TRAINING_DIR, "--data-dir", DATA_DIR,
                       "--split", "test", "--output", output]
            if rates is None:
                command.append("--native")
            else:
                command.extend(["--target-rps", *map(str, rates)])
            run_command(command)
        test_results[label] = output
    for filename, expected in checkpoint_hashes.items():
        assert sha256(TRAINING_DIR / filename) == expected, "A frozen checkpoint changed."
else:
    test_results = {}
    print("Final test scenarios have not been evaluated in this notebook run.")


In [ ]:
scenario_summary = pd.DataFrame()
scenario_runs = pd.DataFrame()
if test_results:
    summaries, detailed = [], []
    for label, output in test_results.items():
        summary = pd.read_csv(output / "summary.csv")
        summary["load_case"] = label
        summaries.append(summary)
        rows = pd.read_csv(output / "runs.csv")
        rows["load_case"] = label
        detailed.append(rows)
    scenario_summary = pd.concat(summaries, ignore_index=True)
    scenario_runs = pd.concat(detailed, ignore_index=True)
    display(scenario_summary)
    scenario_summary.to_csv(RUN_ROOT / "test_scenario_summary.csv", index=False)
    scenario_runs.to_csv(RUN_ROOT / "test_scenario_runs.csv", index=False)

    for label in scenario_summary["load_case"].unique():
        frame = scenario_summary[scenario_summary["load_case"] == label]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
        for ax, metric, title in zip(axes,
            ["overall_slo_attainment", "overall_drop_rate", "overall_goodput_rps"],
            ["Request-weighted SLO attainment", "Request-weighted drop rate", "Goodput (requests/s)"]):
            frame.pivot(index="group", columns="policy", values=metric).plot.bar(ax=ax, rot=0)
            ax.set_title(title)
            ax.set_xlabel("Test scenario")
            ax.set_ylim(bottom=0)
            ax.grid(axis="y", alpha=0.2)
            if metric != "overall_goodput_rps":
                ax.yaxis.set_major_formatter(PercentFormatter(1))
        fig.suptitle("Frozen policies — " + label)
        save_figure(fig, "test_comparison_" + label)
else:
    print("Enable final test evaluation above to populate scenario charts.")


The next chart groups test requests into one-hour blocks by their episode start timestamp, showing request-weighted SLO attainment over time. Missing blocks remain gaps. Episode durations should be at most one hour and divide an hour for this view. Decisions and outcomes are still simulated per episode, with queue resets at boundaries.


In [ ]:
if not scenario_runs.empty and EPISODE_SECONDS <= 3600 and 3600 % EPISODE_SECONDS == 0:
    native = scenario_runs[scenario_runs["load_case"] == "native"].copy()
    groups = list(native["group"].unique())
    fig, axes = plt.subplots(len(groups), 1, figsize=(13, 4 * len(groups)), squeeze=False, constrained_layout=True)
    for ax, group in zip(axes.flat, groups):
        frame = native[native["group"] == group].copy()
        ranges = [item for item in range_config["ranges"] if item["split"] == "test" and item["group"] == group]
        start = min(item["start_timestamp_s"] for item in ranges)
        end = max(item["end_timestamp_s"] for item in ranges)
        frame["hour"] = ((frame["start_timestamp_s"] - start) // 3600).astype(int)
        hourly = frame.groupby(["policy", "hour"])[["arrivals", "slo_met"]].sum()
        hours = range(int(np.ceil((end - start) / 3600)))
        for policy in frame["policy"].unique():
            curve = hourly.loc[policy].reindex(hours)
            ax.plot(curve.index / 24, curve["slo_met"] / curve["arrivals"], linewidth=0.9, label=policy)
        ax.set(title=group + " — frozen policy performance", xlabel="Days from scenario start", ylabel="SLO attainment")
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(alpha=0.2)
        ax.legend()
    save_figure(fig, "test_attainment_over_time")
elif not scenario_runs.empty:
    print("Hourly chart skipped: choose episodes that divide one hour for this view.")


## Share with the team
The export contains charts, result tables, configs, provenance, checkpoints, and logs. It excludes raw CSV and prepared request traces. It also writes a self-contained HTML report with embedded figures, so teammates can view results without Python.

For an executed notebook, use **File → Save a copy in Drive** (retain outputs), then share the Drive link. Download the ZIP below for the accompanying tables and checkpoint evidence. Colab runtime files disappear when the runtime is recycled unless you download them or copy them to Drive.

Single-seed results do not establish statistical significance. Report each seed separately and distinguish validation/model-selection scores from held-out test results.


In [ ]:
SHARE = RUN_ROOT / "share"
SHARE.mkdir(exist_ok=True)
(SHARE / "figures").mkdir(exist_ok=True)
for filename in sorted(generated_figures):
    path = FIGURES / filename
    # Exclude old test figures if test evaluation is disabled on a rerun.
    if path.stem.startswith("test_") and not test_results:
        continue
    shutil.copyfile(path, SHARE / "figures" / path.name)

files_to_copy = {
    RANGE_CONFIG_PATH: "ranges.json",
    TRAIN_CONFIG_PATH: "training_config.json",
    RUN_ROOT / "dataset_summary.csv": "dataset_summary.csv",
    RUN_ROOT / "validation_weighted_summary.csv": "validation_weighted_summary.csv",
    DATA_DIR / "processed" / "manifest.json": "dataset_manifest.json",
    ANALYSIS_DIR / "summary.json": "traffic_analysis_summary.json",
    TRAIN_OUTPUT / "pipeline.json": "pipeline.json",
    TRAIN_OUTPUT / "validation.csv": "validation_runs.csv",
    TRAINING_DIR / "run.json": "training_record.json",
    TRAINING_DIR / "training.csv": "training_history.csv",
    TRAINING_DIR / "checkpoints.csv": "checkpoint_selection.csv",
    COMMAND_LOG: "notebook_commands.log",
}
for path, filename in files_to_copy.items():
    shutil.copyfile(path, SHARE / filename)
for filename in checkpoint_hashes:
    shutil.copyfile(TRAINING_DIR / filename, SHARE / filename)
training_log = TRAIN_OUTPUT / pipeline_state["training_log"]
shutil.copyfile(training_log, SHARE / "training.log")
if test_results:
    shutil.copyfile(RUN_ROOT / "test_scenario_summary.csv", SHARE / "test_scenario_summary.csv")
    shutil.copyfile(RUN_ROOT / "test_scenario_runs.csv", SHARE / "test_scenario_runs.csv")
    for label, output in test_results.items():
        shutil.copyfile(output / "run.json", SHARE / ("test_provenance_" + label + ".json"))

versions = {name: importlib.metadata.version(name) for name in ("numpy", "gymnasium", "pandas", "matplotlib")}
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
source_paths = [REPO / "pyproject.toml", *sorted((REPO / "src").rglob("*.py")),
                *sorted((REPO / "experiments").glob("*.py")), *sorted((REPO / "scripts").glob("*.py"))]
code_hashes = {path.relative_to(REPO).as_posix(): sha256(path) for path in source_paths}
provenance = dict(git_revision=revision, versions=versions, python=sys.version,
                  code_sha256=code_hashes, source_sha256=manifest["source_sha256"],
                  checkpoints=checkpoint_hashes, evaluated_test_cases=list(test_results))
(SHARE / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")

status = "Frozen held-out tests included" if test_results else "Validation only — final test scenarios not evaluated"
parts = [
    "<!doctype html><html><head><meta charset='utf-8'><title>BurstGPT 3 experiment</title>",
    "<style>body{font:16px system-ui;max-width:1200px;margin:32px auto;padding:0 20px;color:#17202a}"
    "img{max-width:100%}table{border-collapse:collapse;font-size:13px;display:block;overflow-x:auto}"
    "td,th{padding:8px;border:1px solid #ddd}pre{white-space:pre-wrap}h2{margin-top:32px}</style></head><body>",
    "<h1>BurstGPT 3 experiment</h1><p>" + html.escape(status) + "</p>",
    "<p>Simulated GPU service; native traffic retains source timing. Episode queues reset at boundaries. "
    "The initial budget is exploratory. Checkpoint selection uses the mean episode validation score; "
    "comparison tables also show request-weighted scores. No online adaptation is performed.</p>",
    "<h2>Dataset</h2>", dataset_summary.to_html(index=False),
    "<h2>Training settings</h2><pre>" + html.escape(json.dumps(train_config, indent=2)) + "</pre>",
    "<h2>Selected checkpoints</h2>", pd.DataFrame(training_record["checkpoints"]).to_html(index=False),
    "<h2>Validation comparison</h2>", validation_summary.to_html(index=False),
]
if test_results:
    parts.extend(["<h2>Frozen test comparisons</h2>", scenario_summary.to_html(index=False),
                  "<p>Native goodput includes empty-episode duration. Scaled evaluations exclude empty-episode "
                  "duration because scaling an episode without arrivals is undefined. No-record periods may be collection gaps.</p>"])
for filename in sorted(generated_figures):
    image_path = SHARE / "figures" / filename
    if image_path.stem.startswith("test_") and not test_results:
        continue
    encoded = base64.b64encode(image_path.read_bytes()).decode("ascii")
    parts.append("<h2>" + html.escape(image_path.stem.replace("_", " ")) + "</h2>"
                 "<img alt='" + html.escape(image_path.stem) + "' src='data:image/png;base64," + encoded + "'>")
parts.extend(["<h2>Provenance</h2><pre>" + html.escape(json.dumps(provenance, indent=2)) + "</pre></body></html>"])
(SHARE / "report.html").write_text("\n".join(parts), encoding="utf-8")

# Only include the explicitly selected report files, never raw/processed data.
members = set(files_to_copy.values()) | set(checkpoint_hashes) | {"training.log", "provenance.json", "report.html"}
members |= {"figures/" + name for name in generated_figures
            if test_results or not name.startswith("test_")}
if test_results:
    members |= {"test_scenario_summary.csv", "test_scenario_runs.csv"}
    members |= {"test_provenance_" + label + ".json" for label in test_results}
ARCHIVE = RUN_ROOT / (run_name + "_team_results.zip")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in sorted(members):
        archive.write(SHARE / name, name)
print(status)
display(FileLink(str(SHARE / "report.html")))
display(FileLink(str(ARCHIVE)))
print("ZIP size:", round(ARCHIVE.stat().st_size / 1024**2, 2), "MB")


In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))
else:
    print("Team report:", SHARE / "report.html")
    print("Results archive:", ARCHIVE)

# Optional after mounting Drive:
# shutil.copyfile(ARCHIVE, Path("/content/drive/MyDrive") / ARCHIVE.name)
